In [0]:
import json

account    = "gs1datalake"
container  = "datalake"

import unicodedata, re

access_key = dbutils.secrets.get("gs1-kv", "storage-access-key")

allowed = set("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=")
bad = [(i, ord(c), unicodedata.name(c, "UNKNOWN"))
       for i, c in enumerate(access_key) if c not in allowed]

print("bad_count:", len(bad))
for i, code, name in bad:
    print(f"idx={i}, U+{code:04X}, {name}")



spark.conf.set(f"fs.azure.account.key.{account}.dfs.core.windows.net", access_key)

DELTA_PRODUCTS = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/silver/catalogue_items"

# 🔹 Lue Silverin Delta-taulu
silver_df = spark.read.format("delta").load(DELTA_PRODUCTS)

# 🔹 Haetaan useampi rivi
rows = silver_df.select("Id", "raw_json").limit(4).collect()

# 🔹 Käydään läpi ja tulostetaan kaikki 3 tuotetta
for i, row in enumerate(rows, start=1):
    data = json.loads(row["raw_json"])
    print(f"\n================ TUOTE {i} ================\n")
    print(f"Tuotteen Id: {row['Id']}\n")
    print(json.dumps(data, indent=4, ensure_ascii=False))
